[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/padillacm/IA-practice/blob/master/agent-observability-course/08_loki_logql/08_proyecto_loki_logql.ipynb)

# Proyecto 08 · Logs de SupportOps correlacionados con trazas: del formato a la investigación

| | |
|---|---|
| **Nivel** | 🟡 Intermedio |
| **Duración estimada** | ≈ 1 h 45 min |
| **Servicios de Compose** | Opcionales (`core`) para la verificación contra Loki; la corrección usa el mini motor LogQL y funciona **sin Docker** (Colab) |
| **RAM aproximada** | < 500 MB (+ ≈ 0,4 GB con el stack) |
| **Prerrequisitos** | Lección `08_loki_logql` y proyecto 07 (el `trace_id` que enlaza ambos mundos) |
| **Conexión** | Cierra el ciclo «traza → logs → causa» del hilo conductor. El formato de log que diseñes es el que usarán el módulo 20 (redacción de PII) y el 21 (mesa de incidentes) |

## Contexto de negocio

Eres ingeniero/a de plataforma en **SupportOps**. Los logs actuales son texto suelto y nadie puede saltar de una queja de cliente a la traza. Debes (1) **definir el formato de log** del agente (JSON con el modelo de datos de logs de OpenTelemetry, contexto de traza y **sin correos** en el mensaje), (2) **convertir** cada log a una entrada de Loki con **etiquetas bien elegidas**, (3) responder **cuatro preguntas de investigación** con LogQL y (4) preparar dos consultas de métricas y un recuento por minuto.

## Datos

El lote pregrabado `recorded/logs_seed42.json.gz`: **660 líneas** en 4 streams (`supportops-agent-json`, `llm-gateway-access`, `orders-api-access`, `kb-search-access`) del mismo lote que usaste en el módulo 07, con `trace_id` como *structured metadata*, y la verdad de campo `RES`. La corrección usa `logql_mini` (validado contra Loki 3.7 en la lección). Con el stack, una celda final verifica tus consultas en Loki real.

## Entregables y criterios de evaluación (rúbrica)

| # | Entregable | Criterio medido | Umbral (calibrado con la solución, lote pregrabado) |
|---|---|---|---|
| R1 | `OtelJsonFormatter` | 5 casos: INFO sin span (sin `trace_id`); WARNING dentro de un span (`WARN`, `trace_id`/`span_id` correctos, extras en `attributes`); correo en el mensaje → `[email]`; excepción en **una sola línea** con `exception.stacktrace`; CRITICAL → `FATAL` | 5/5 |
| R2 | `to_loki_entry` | Etiquetas **exactamente** `service_name` y `deployment_environment_name`; metadatos con `trace_id`/`span_id`/`ticket_id`; marca de tiempo en ns correcta; línea = JSON | 4/4 comprobaciones |
| R3 | `INVESTIGATION` I1-I4 | Nº de líneas y huella (sha1 de las marcas de tiempo) de cada consulta | 4/4 exactas |
| R4 | `errors_per_minute` | Lista de 10 enteros: logs de acceso con `status >= 500` por minuto desde la primera línea: `[1, 0, 6, 17, 7, 4, 2, 4, 0, 0]` | exacto |
| R5 | `METRICS` | 2 consultas de métricas (tasa de 5xx por ruta; p95 de `duration_ms` por servicio con `unwrap`) con la sintaxis esperada | 2/2 |
| R6 | Verificación en vivo (opcional) | Con Loki: I1, I2 y I4 devuelven el mismo nº de líneas que el mini motor sobre un lote nuevo | 3/3 (se omite sin stack) |

> 📏 Calibración: los umbrales son lo que mide la solución sobre el lote pregrabado (semilla 42, tiempo simulado); nada depende de tu máquina.

In [ ]:
# Instala el paquete del curso (editable) con extras de gráficos. Sirve en local y en Colab (clona el repo si hace falta).
import pathlib, subprocess, sys

def _find_root():
    for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (p / "pyproject.toml").exists() and (p / "supportops").is_dir():
            return p
    return None

ROOT = _find_root()
if ROOT is None:  # p. ej. Google Colab: el notebook se abre sin el repo
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/padillacm/IA-practice", "/content/IA-practice"], check=True)
    ROOT = pathlib.Path("/content/IA-practice/agent-observability-course")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{ROOT}[viz,otel]"], check=True)
print("Curso en:", ROOT)

In [ ]:
import os
os.environ.setdefault("LLM_PROVIDER", "mock")          # mock = determinista, sin coste, sin API keys
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from supportops import *                                 # SupportAgent, generate_tickets, check_stack, ...
from supportops import viz, sample_data, schema, backends
viz.apply_style()
pd.set_option("display.width", 140); pd.set_option("display.max_colwidth", 80)
SEED = 42
FAST_DEV_RUN = True   # True: pocos tickets (iteración rápida). False: tamaños «completos» del texto

import json, gzip, re, time, collections, inspect, logging, io
from supportops import backends, logql_mini as lq, loki_lab as ll
from supportops.loki_lab import LokiPusher, run_logs_batch, wait_for_logs, loki_range, loki_stats, logfmt
STACK = check_stack("core")
HAVE_LOKI = bool(STACK.up.get("loki")); HAVE_TEMPO = bool(STACK.up.get("tempo")); HAVE_GRAFANA = bool(STACK.up.get("grafana"))
print(f"\nLoki en vivo: {HAVE_LOKI} · Tempo: {HAVE_TEMPO} · Grafana: {HAVE_GRAFANA}  ->  ",
      "modo COMPLETO" if HAVE_LOKI and HAVE_TEMPO else ("modo parcial (sin Tempo)" if HAVE_LOKI else "MODO DEGRADADO (datos y capturas pregrabados)"))

In [ ]:
REC = ROOT / "08_loki_logql" / "recorded"
DATA = json.load(gzip.open(REC / "logs_seed42.json.gz", "rt", encoding="utf-8"))     # logs (3 rutas de ingesta) y trazas de un lote pregrabado: 120 tickets, semilla 42
ENTRIES = lq.entries_from_payload(DATA["payload"]); RES = pd.DataFrame(DATA["results"]); SPANS = DATA["spans"]
print(f"{len(ENTRIES)} líneas en {len({e.labels['service_name'] for e in ENTRIES})} streams · {RES.shape[0]} tickets")

## Paso 0 · El material

Cada `Entry` de `ENTRIES` tiene `ts_ns`, `line`, `labels` (índice) y `metadata` (structured metadata). `logql.query(ENTRIES, '...')` evalúa una consulta de logs (mini motor). Explora antes de escribir nada.

In [ ]:
import hashlib, datetime
from opentelemetry import trace as otel_trace
from supportops.telemetry import setup_telemetry
def run_q(q, entries=ENTRIES):
    return lq.query(entries, q)
def fp(hits):
    return [len(hits), hashlib.sha1(",".join(str(e.ts_ns) for e in sorted(hits, key=lambda e: e.ts_ns)).encode()).hexdigest()[:10]]
for s_ in sorted({e.labels["service_name"] for e in ENTRIES}):
    e = next(e for e in ENTRIES if e.labels["service_name"] == s_); print(f"{s_:24s} {sum(1 for x in ENTRIES if x.labels['service_name']==s_):4d} líneas · ejemplo: {e.line[:105]}")

## Paso 1 · El formato de log (logging estructurado)

**TODO 1.** Implementa `OtelJsonFormatter(logging.Formatter)`: `format(record)` devuelve **una línea JSON** con las claves del modelo de logs de OpenTelemetry:

| Clave | Valor |
|---|---|
| `ts` | instante de `record.created` en ISO-8601 UTC con milisegundos |
| `severity_text` | `DEBUG`, `INFO`, `WARN`, `ERROR` o `FATAL` (nombres de OpenTelemetry: `WARNING→WARN`, `CRITICAL→FATAL`) |
| `body` | el mensaje (`record.getMessage()`) con **cualquier correo sustituido por `[email]`** (regex `[\w.+-]+@[\w-]+\.[\w.]+`) |
| `attributes` | dict con los campos extra pasados con `extra={...}` (`ticket_id`, `status`...) y, si hay excepción, `exception.stacktrace` con el *traceback* en el propio valor |
| `trace_id`, `span_id` | del **span activo** (hex de 32 y de 16 caracteres) **solo si hay uno** |

<details>
<summary>🪜 Pista 1</summary>

Los atributos «de serie» de `LogRecord` (`name`, `msg`, `args`, `levelname`, `levelno`, `pathname`, `filename`, `module`, `exc_info`, `exc_text`, `stack_info`, `lineno`, `funcName`, `created`, `msecs`, `relativeCreated`, `thread`, `threadName`, `processName`, `process`, `message`, `taskName`) no son extras: los extras son el resto de `record.__dict__`.

</details>

<details>
<summary>🪜 Pista 2</summary>

`sc = otel_trace.get_current_span().get_span_context()`; `sc.is_valid`; `format(sc.trace_id, '032x')`, `format(sc.span_id, '016x')`. El traceback: `self.formatException(record.exc_info)`; al ir dentro de JSON los saltos de línea se escapan, así que la línea sigue siendo única.

</details>

In [ ]:
EMAIL_RE = re.compile(r"[\w.+-]+@[\w-]+\.[\w.]+")
SEVERITY = {"DEBUG": "DEBUG", "INFO": "INFO", "WARNING": "WARN", "ERROR": "ERROR", "CRITICAL": "FATAL"}

class OtelJsonFormatter(logging.Formatter):
    def format(self, record):
        # TODO 1
        raise NotImplementedError("TODO 1")

def demo_logger(fmt):
    buf = io.StringIO(); h = logging.StreamHandler(buf); h.setFormatter(fmt)
    lg = logging.getLogger(f"demo.p08.{id(buf)}"); lg.handlers[:] = [h]; lg.setLevel(logging.DEBUG); lg.propagate = False
    return lg, buf

try:
    lg, buf = demo_logger(OtelJsonFormatter())
    tel = setup_telemetry("demo", mode="memory", set_global=False)
    with tel.tracer("demo").start_as_current_span("invoke_agent") as sp:
        lg.warning("el cliente ana@example.com pregunta por su pedido", extra={"ticket_id": "T-1", "status": 429})
    tel.shutdown(); print(buf.getvalue().strip())
except NotImplementedError as e:
    print("Pendiente:", e)

## Paso 2 · De log a entrada de Loki

**TODO 2.** `to_loki_entry(doc, service="supportops-agent-json", env="lab")` recibe el **dict** que produce tu formateador y devuelve la tupla `(ts_ns, line, labels, metadata)` que espera `LokiPusher.add`:
- `ts_ns`: `int` de nanosegundos del campo `ts` (ISO);
- `line`: `json.dumps(doc, ensure_ascii=False)`;
- `labels`: **exactamente** `{"service_name": service, "deployment_environment_name": env}` — nada de `trace_id`, `ticket_id` ni nivel como etiquetas (cardinalidad);
- `metadata`: `trace_id`, `span_id` (si existen en `doc`) y `ticket_id` (si está en `doc["attributes"]`), todo `str`.

<details>
<summary>🪜 Pista 1</summary>

`datetime.datetime.fromisoformat(doc['ts'])` devuelve un `datetime` con zona; `int(dt.timestamp() * 1e9)` pierde precisión de nanosegundos por el `float`: usa `int(dt.timestamp()) * 10**9 + dt.microsecond * 1000`.

</details>

In [ ]:
def to_loki_entry(doc: dict, service: str = "supportops-agent-json", env: str = "lab"):
    # TODO 2
    raise NotImplementedError("TODO 2")
try:
    d = {"ts": "2026-10-15T12:00:00.123+00:00", "severity_text": "WARN", "body": "x", "attributes": {"ticket_id": "T-9"}, "trace_id": "a" * 32, "span_id": "b" * 16}
    print(to_loki_entry(d))
except NotImplementedError as e:
    print("Pendiente:", e)

## Paso 3 · Cuatro preguntas de investigación (LogQL)

**TODO 3.** Rellena `INVESTIGATION` con consultas de **logs** (evaluadas con `logql_mini`):

| # | Pregunta |
|---|---|
| I1 | Avisos (`warn`) del **agente** (stream JSON) — usa el parser `json` y un filtro de etiqueta |
| I2 | Peticiones al **gateway LLM** (`llm-gateway-access`) con HTTP **5xx** — `logfmt` y filtro numérico |
| I3 | **Todos los logs** (agente y servicios, sin los de OTLP) de la petición del ticket **`T-4200004`**, a partir de su `trace_id` (`TRACE_ID` ya calculado) — structured metadata, sin parsear |
| I4 | Llamadas a **`orders-api`** **correctas** (`status = 200`) que tardaron **más de 300 ms** |

In [ ]:
TRACE_ID = DATA["trace_ids"]["T-4200004"]
INVESTIGATION: dict[str, str] = {
    # "I1": '{service_name="..."} | ...',
}
for k, q in INVESTIGATION.items():
    try:
        print(k, fp(run_q(q)), q)
    except Exception as e:
        print(k, "ERROR", type(e).__name__, e)

## Paso 4 · La forma temporal y las métricas

**TODO 4a.** `errors_per_minute(entries, n_bins=10)`: lista de `n_bins` enteros con el nº de líneas de los streams `*-access` con **`status >= 500`** por intervalo de 60 s desde la marca de tiempo de la **primera línea del lote** (`t0 = min(e.ts_ns)`); lo posterior al último intervalo cuenta en el último. Parsea con `lq.parse_logfmt(e.line)`.

**TODO 4b.** Escribe en `METRICS` dos consultas de **métricas LogQL**: `"err_rate"`: tasa de 5xx **por ruta** (`sum by (route)`, `rate`, ventana de 1 m) y `"p95"`: p95 de `duration_ms` **por servicio** (`quantile_over_time(0.95, ... | unwrap duration_ms [2m]) by (service_name)`), ambas sobre los logs `*-access` con `logfmt`.

In [ ]:
def errors_per_minute(entries, n_bins=10):
    # TODO 4a
    raise NotImplementedError("TODO 4a")
METRICS: dict[str, str] = {
    # "err_rate": '...',
}
try:
    EPM = errors_per_minute(ENTRIES); print(EPM)
    fig, ax = plt.subplots(figsize=(8, 3)); ax.bar(range(len(EPM)), EPM, color=viz.PALETTE["error"]); ax.set_xlabel("minuto"); ax.set_ylabel("5xx"); ax.set_title("5xx en los logs de acceso"); plt.show()
except NotImplementedError as e:
    print("Pendiente:", e)
for k, q in METRICS.items(): print(k, "→", q)

## Autocomprobación (rúbrica)

In [ ]:
EXPECTED = {'I1': [73, 'ae65d50444'], 'I2': [32, '4a85a07b13'], 'I3': [8, '5e7147de48'], 'I4': [13, '35c3a13bf8']}
def check(name, fn):
    try:
        ok, detail = fn()
    except NotImplementedError:
        ok, detail = False, "pendiente"
    except Exception as e:
        ok, detail = False, f"{type(e).__name__}: {e}"
    print(("✅" if ok else "❌"), name, "·", detail); return ok
def r1():
    tel = setup_telemetry("p08", mode="memory", set_global=False); tr = tel.tracer("p08"); res = []
    try:
        lg, buf = demo_logger(OtelJsonFormatter())
        lg.info("fuera de span", extra={"ticket_id": "T-1"}); a = json.loads(buf.getvalue().strip().splitlines()[-1])
        res.append(a["severity_text"] == "INFO" and "trace_id" not in a and a["attributes"].get("ticket_id") == "T-1")
        with tr.start_as_current_span("s") as sp:
            lg.warning("aviso", extra={"ticket_id": "T-2", "status": 429}); b = json.loads(buf.getvalue().strip().splitlines()[-1])
            sc = sp.get_span_context()
            res.append(b["severity_text"] == "WARN" and b["trace_id"] == format(sc.trace_id, "032x") and b["span_id"] == format(sc.span_id, "016x") and b["attributes"].get("status") == 429)
            lg.error("el cliente ana@example.com escribió", extra={"ticket_id": "T-3"}); c = json.loads(buf.getvalue().strip().splitlines()[-1])
            res.append(c["body"] == "el cliente [email] escribió")
            try:
                1 / 0
            except ZeroDivisionError:
                lg.exception("fallo"); raw = buf.getvalue().strip().splitlines()[-1]
            d = json.loads(raw); res.append("ZeroDivisionError" in d["attributes"]["exception.stacktrace"] and "\n" not in raw and d["severity_text"] == "ERROR")
        lg.critical("fin"); res.append(json.loads(buf.getvalue().strip().splitlines()[-1])["severity_text"] == "FATAL")
    finally:
        tel.shutdown()
    return sum(res) == 5, f"{sum(res)}/5 casos"
def r2():
    doc = {"ts": "2026-10-15T12:00:00.123456+00:00", "severity_text": "WARN", "body": "x", "attributes": {"ticket_id": "T-9", "status": 429}, "trace_id": "a" * 32, "span_id": "b" * 16}
    ts, line, labels, meta = to_loki_entry(doc)
    ok = [labels == {"service_name": "supportops-agent-json", "deployment_environment_name": "lab"}, meta == {"trace_id": "a" * 32, "span_id": "b" * 16, "ticket_id": "T-9"},
          ts == 1792065600123456000, json.loads(line) == doc]
    return all(ok), f"{sum(ok)}/4 comprobaciones {ok}"
def r3():
    good = [k for k in EXPECTED if k in INVESTIGATION and fp(run_q(INVESTIGATION[k])) == EXPECTED[k]]
    return len(good) == 4, f"{len(good)}/4 exactas: {good}"
def r4():
    return errors_per_minute(ENTRIES) == [1, 0, 6, 17, 7, 4, 2, 4, 0, 0], f"{errors_per_minute(ENTRIES)}"
def r5():
    a, b = METRICS.get("err_rate", ""), METRICS.get("p95", "")
    ok_a = "sum by (route)" in a and "rate(" in a and "logfmt" in a and "status >= 500" in a and "[1m]" in a and "-access" in a
    ok_b = "quantile_over_time(0.95" in b and "unwrap duration_ms" in b and "[2m]" in b and "by (service_name)" in b and "logfmt" in b
    return ok_a and ok_b, f"err_rate {ok_a} · p95 {ok_b}"
def r6():
    if not HAVE_LOKI:
        return True, "omitido (sin Loki)"
    B, P = run_logs_batch(120, 42, live=True); mine = set(B.trace_ids.values()); es = lq.entries_from_payload(P.full_payload())
    n_json = sum(1 for e in es if e.labels["service_name"] == "supportops-agent-json"); wait_for_logs('{service_name="supportops-agent-json"}', int(0.98 * n_json), 60, verbose=False)
    same = 0
    for k in ("I1", "I2", "I4"):
        live = sum(1 for x in backends.loki_query(INVESTIGATION[k], minutes=30, limit=5000) if x["labels"].get("trace_id") in mine)
        same += live == len(lq.query(es, INVESTIGATION[k]))
    return same == 3, f"{same}/3 consultas devuelven el mismo nº de líneas en Loki y en el mini motor"
results = [check("R1 formateador", r1), check("R2 to_loki_entry", r2), check("R3 investigación", r3), check("R4 errors_per_minute", r4), check("R5 métricas", r5), check("R6 verificación en vivo", r6)]
print(f"\n{sum(results)}/6 entregables correctos")

---
# ⛔ SPOILER — intenta resolverlo primero

Solución de referencia completa (con la que se calibró la rúbrica). Si te atascas, abre antes las pistas 🪜.

In [ ]:
class OtelJsonFormatter(logging.Formatter):
    RESERVED = {"name", "msg", "args", "levelname", "levelno", "pathname", "filename", "module", "exc_info", "exc_text", "stack_info", "lineno", "funcName",
                "created", "msecs", "relativeCreated", "thread", "threadName", "processName", "process", "message", "taskName"}
    def format(self, record):
        attrs = {k: v for k, v in record.__dict__.items() if k not in self.RESERVED}
        if record.exc_info:
            attrs["exception.stacktrace"] = self.formatException(record.exc_info)
        doc = {"ts": datetime.datetime.fromtimestamp(record.created, datetime.timezone.utc).isoformat(timespec="milliseconds"),
               "severity_text": SEVERITY.get(record.levelname, "INFO"), "body": EMAIL_RE.sub("[email]", record.getMessage()), "attributes": attrs}
        sc = otel_trace.get_current_span().get_span_context()
        if sc.is_valid:
            doc["trace_id"], doc["span_id"] = format(sc.trace_id, "032x"), format(sc.span_id, "016x")
        return json.dumps(doc, ensure_ascii=False, default=str)

def to_loki_entry(doc: dict, service: str = "supportops-agent-json", env: str = "lab"):
    dt = datetime.datetime.fromisoformat(doc["ts"])
    ts_ns = int(dt.timestamp()) * 10**9 + dt.microsecond * 1000
    meta = {k: str(doc[k]) for k in ("trace_id", "span_id") if doc.get(k)}
    if doc.get("attributes", {}).get("ticket_id"):
        meta["ticket_id"] = str(doc["attributes"]["ticket_id"])
    return ts_ns, json.dumps(doc, ensure_ascii=False), {"service_name": service, "deployment_environment_name": env}, meta

INVESTIGATION = {
    "I1": '{service_name="supportops-agent-json"} | json | level="warn"',
    "I2": '{service_name="llm-gateway-access"} | logfmt | status >= 500',
    "I3": '{service_name=~"supportops-agent-json|.+-access"} | trace_id="%s"' % TRACE_ID,
    "I4": '{service_name="orders-api-access"} | logfmt | duration_ms > 300 and status = 200',
}
def errors_per_minute(entries, n_bins=10):
    t0 = min(e.ts_ns for e in entries); out = [0] * n_bins
    for e in entries:
        if e.labels["service_name"].endswith("-access") and int(lq.parse_logfmt(e.line)["status"]) >= 500:
            out[min(n_bins - 1, int((e.ts_ns - t0) / 60e9))] += 1
    return out
METRICS = {
    "err_rate": 'sum by (route) (rate({service_name=~".+-access"} | logfmt | status >= 500 [1m]))',
    "p95": 'quantile_over_time(0.95, {service_name=~".+-access"} | logfmt | unwrap duration_ms [2m]) by (service_name)',
}
print(fp(run_q(INVESTIGATION["I3"])), errors_per_minute(ENTRIES))

In [ ]:
results = [check("R1 formateador", r1), check("R2 to_loki_entry", r2), check("R3 investigación", r3), check("R4 errors_per_minute", r4), check("R5 métricas", r5), check("R6 verificación en vivo", r6)]
print(f"\n{sum(results)}/6 entregables correctos")

## Retos extra (nivel experto)

1. **Envíalo de verdad:** con el stack, genera un lote con `OtelJsonFormatter` + `to_loki_entry` + `LokiPusher` (un `Instrumentor` propio sobre `log(...)`) y verifica con `loki_stats` que tu pipeline procesa los mismos bytes que `JsonLogInstrumentor`.
2. **Presupuesto de etiquetas:** calcula el nº de streams si añadieras `level` y `tier` como etiquetas a los 4 servicios; ¿cuánto crece el nº de chunks con el volumen de este lote?
3. **Alerta puente:** escribe una regla del *ruler* (YAML) con `sum(rate({service_name=~".+-access"} | logfmt | status >= 500 [5m])) > 0.05` y explica cuándo la sustituirías por una métrica.
4. **Redacción mejor:** amplía el formateador para ocultar también teléfonos (`+34 6xx xxx xxx`); luego mueve la redacción al Collector (procesador `transform`, módulo 20) y razona qué política es más robusta.
5. **Línea de tiempo:** reproduce el gráfico de la sección 6 para el ticket con más reintentos del lote.

## Reflexión

- ¿Qué campos de tus logs deberían ser etiquetas, cuáles metadatos y cuáles contenido, y qué cambiaría si SupportOps pasara de 4 a 400 servicios?
- Un auditor te pide «todos los logs de este cliente en los últimos 30 días». ¿Qué atributo falta como metadato para poder responder sin parsear todo? (módulo 20)
- ¿Qué señales que hoy salen de logs (`rate` de 5xx, p95 con `unwrap`) migrarías a métricas en el módulo 09 y por qué?
- Conexión con SupportOps: el `trace_id` en cada línea es la base de los paneles de correlación del módulo 10 y de las alertas con enlace a traza del 16.